# Exploratory Data Analysis
Understand the shape and relationships in the data before modeling.

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

teams = pd.read_csv('../data_clean/teams_clean.csv')
games = pd.read_csv('../data_clean/games_clean.csv', parse_dates=['Date'])

print(teams.shape)
teams.head()

## 1. Distribution of Win Totals

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(teams[teams['Conference']=='East']['wins'], bins=10, alpha=0.7, label='East', color='#1d428a')
ax.hist(teams[teams['Conference']=='West']['wins'], bins=10, alpha=0.7, label='West', color='#c8102e')
ax.set_xlabel('Wins')
ax.set_ylabel('Teams')
ax.set_title('Distribution of Wins by Conference')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Correlation Heatmap

In [ ]:
import matplotlib.colors as mcolors

num_cols = ['wins', 'win_pct', 'point_diff', 'srs', 'NRtg', 'ORtg', 'DRtg', 'Pace', 'SOS']
corr = teams[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='right')
ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Correlation Matrix — Team Metrics')
plt.tight_layout()
plt.show()

## 3. Net Rating vs Win %

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for conf, grp in teams.groupby('Conference'):
    color = '#1d428a' if conf == 'East' else '#c8102e'
    ax.scatter(grp['NRtg'], grp['win_pct'], color=color, label=conf, s=60, zorder=3)
    for _, row in grp.iterrows():
        ax.annotate(row['Team'].split()[-1], (row['NRtg'], row['win_pct']),
                    xytext=(4, 2), textcoords='offset points', fontsize=7)
# best-fit line
x = teams['NRtg']
m, b = np.polyfit(x, teams['win_pct'], 1)
ax.plot(sorted(x), [m*xi+b for xi in sorted(x)], '--', color='#888888', linewidth=1)
ax.set_xlabel('Net Rating (points per 100 possessions)')
ax.set_ylabel('Win %')
ax.set_title('Net Rating vs Win Percentage')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Offensive vs Defensive Rating

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for conf, grp in teams.groupby('Conference'):
    color = '#1d428a' if conf == 'East' else '#c8102e'
    sc = ax.scatter(grp['ORtg'], grp['DRtg'], color=color, label=conf,
                    s=grp['wins']*2, alpha=0.8, zorder=3)
    for _, row in grp.iterrows():
        ax.annotate(row['Team'].split()[-1], (row['ORtg'], row['DRtg']),
                    xytext=(4, 2), textcoords='offset points', fontsize=7)
ax.axhline(teams['DRtg'].mean(), color='#888', linestyle='--', linewidth=1)
ax.axvline(teams['ORtg'].mean(), color='#888', linestyle='--', linewidth=1)
ax.invert_yaxis()  # lower DRtg = better defense
ax.set_xlabel('Offensive Rating (higher = better)')
ax.set_ylabel('Defensive Rating (lower = better)')
ax.set_title('Offensive vs Defensive Rating\n(bubble size = wins)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Average Game Margin Over Time (League-wide)

In [ ]:
daily_margin = games.groupby('Date')['margin'].mean().rolling(14).mean()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily_margin.index, daily_margin.values, color='#444', linewidth=1.5)
ax.axhline(0, color='#aaa', linestyle='--')
ax.set_xlabel('Date')
ax.set_ylabel('Avg Home Margin (14-game rolling)')
ax.set_title('Home Court Advantage Trend — 2025-26 Season')
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Summary Stats

In [ ]:
print('--- Conference Averages ---')
print(teams.groupby('Conference')[['wins','win_pct','NRtg','ORtg','DRtg','point_diff']].mean().round(2))
print()
print('--- Top 5 by Net Rating ---')
print(teams.nlargest(5, 'NRtg')[['Team','NRtg','wins','win_pct']])